# Projet 2 : Analyse Quantitative du Risque de Marché — Cas d'Étude NVIDIA

**Auteur :** ISLEYEN Volkan  
**Contexte :** Master Finance de Marché · Portfolio Data Finance  
**Sujet :** Modélisation et visualisation de la volatilité historique

---

## Description du Projet

Ce projet analyse la **dynamique de la volatilité** de l'action NVIDIA (*NVDA*) sur la période 2022–2024, marquée par l'essor de l'Intelligence Artificielle. Plutôt qu'une analyse statique, nous implémentons une approche par **fenêtre glissante** (*Rolling Window*) pour capturer l'évolution temporelle du risque et identifier les changements de régime de marché.

## Objectifs

1. **Extraction** — Récupérer les données OHLCV de NVDA via `yfinance`
2. **Rendements** — Calculer les rendements journaliers et cumulés
3. **Volatilité** — Modéliser la volatilité rolling 30 / 60 jours annualisée
4. **Distribution** — Analyser la distribution des rendements (asymétrie, queues)
5. **Visualisation** — Superposer cours, rendements et risque sur un même graphique
6. **Synthèse** — Identifier les régimes de marché (stress vs consolidation)

---

> **Prérequis :** `pip install yfinance pandas matplotlib seaborn numpy`

**Stack :** Python 3.12 · pandas · yfinance · matplotlib · seaborn · numpy

In [ ]:
# ── Imports & configuration ─────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

# Paramètres globaux
TICKER = 'NVDA'
START  = '2022-01-01'
END    = '2024-12-31'
COLOR_NVDA   = '#76b900'   # Vert NVIDIA
COLOR_VOL30  = '#f5a623'   # Amber — fenêtre courte
COLOR_VOL60  = '#ff4d4d'   # Rouge — fenêtre longue

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

print(f"Actif    : {TICKER}")
print(f"Période  : {START} → {END}")

## 1. Importation et préparation des données

In [ ]:
# Téléchargement des données NVDA
nvda = yf.download(TICKER, start=START, end=END, auto_adjust=True, progress=False)
nvda.columns = [col[0] if isinstance(col, tuple) else col for col in nvda.columns]
nvda.dropna(inplace=True)

print(f"Nombre de jours de trading : {len(nvda)}")
print(f"Prix de départ (Jan 2022)  : ${nvda['Close'].iloc[0]:.2f}")
print(f"Prix de fin    (Déc 2024)  : ${nvda['Close'].iloc[-1]:.2f}")
perf_totale = ((nvda['Close'].iloc[-1] / nvda['Close'].iloc[0]) - 1) * 100
print(f"Performance totale         : {perf_totale:+.1f}%")
nvda.head()

## 2. Calcul des rendements

Le **rendement journalier** mesure la variation relative du prix entre deux séances consécutives :

$$R_t = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1$$

Le **rendement cumulé** mesure la performance globale depuis le début de la période :

$$RC_t = \prod_{i=1}^{t}(1 + R_i) - 1$$

In [ ]:
# Rendements journaliers
nvda['Return'] = nvda['Close'].pct_change()

# Rendements cumulés
nvda['Cumulative_Return'] = (1 + nvda['Return']).cumprod() - 1

# Statistiques des rendements
r = nvda['Return'].dropna()
print("Statistiques des rendements journaliers :")
print(f"  Moyenne       : {r.mean()*100:.3f}%")
print(f"  Écart-type    : {r.std()*100:.3f}%")
print(f"  Meilleur jour : {r.max()*100:+.2f}%  ({r.idxmax().strftime('%d/%m/%Y')})")
print(f"  Pire jour     : {r.min()*100:+.2f}%  ({r.idxmin().strftime('%d/%m/%Y')})")
print(f"  Asymétrie     : {r.skew():.3f}")
print(f"  Kurtosis      : {r.kurtosis():.3f}")

## 3. Modélisation de la volatilité

La **volatilité historique** est calculée comme l'écart-type des rendements sur une fenêtre glissante, puis **annualisée** par le facteur $\sqrt{252}$ (nombre de jours de trading par an) :

$$\sigma_{annuelle} = \sigma_{journalière} \times \sqrt{252}$$

Nous calculons deux fenêtres :
- **Rolling 30 jours** : réagit rapidement aux chocs de marché
- **Rolling 60 jours** : filtre le bruit, capture les tendances de fond

In [ ]:
# Volatilité rolling annualisée
nvda['Vol_30']  = nvda['Return'].rolling(window=30).std() * np.sqrt(252)
nvda['Vol_60']  = nvda['Return'].rolling(window=60).std() * np.sqrt(252)

# Statistiques de la volatilité
print("Volatilité annualisée — fenêtre 60 jours :")
print(f"  Moyenne  : {nvda['Vol_60'].mean()*100:.1f}%")
print(f"  Maximum  : {nvda['Vol_60'].max()*100:.1f}%  ({nvda['Vol_60'].idxmax().strftime('%d/%m/%Y')})")
print(f"  Minimum  : {nvda['Vol_60'].min()*100:.1f}%  ({nvda['Vol_60'].idxmin().strftime('%d/%m/%Y')})")

## 4. Évolution du cours NVIDIA (2022–2024)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(nvda.index, nvda['Close'], color=COLOR_NVDA, linewidth=1.8, label='NVDA Close')
ax.fill_between(nvda.index, nvda['Close'], alpha=0.08, color=COLOR_NVDA)

# Annotation du pic
idx_max = nvda['Close'].idxmax()
ax.annotate(
    f"Pic : ${nvda['Close'].max():.0f}\n({idx_max.strftime('%b %Y')})",
    xy=(idx_max, nvda['Close'].max()),
    xytext=(idx_max - pd.Timedelta(days=120), nvda['Close'].max() * 0.85),
    arrowprops=dict(arrowstyle='->', color='white', lw=1.2),
    fontsize=9, color='white'
)

ax.set_title('NVIDIA (NVDA) — Évolution du cours de clôture (2022–2024)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Prix de clôture ($)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Tableau de bord : Prix, Rendements & Volatilité

Ce graphique combiné est l'outil central de l'analyse de risque : il permet de corréler visuellement les pics de volatilité avec les événements de marché.

In [ ]:
fig = plt.figure(figsize=(14, 12))
gs  = gridspec.GridSpec(3, 1, hspace=0.35)

# ── Panel 1 : Prix de clôture ──
ax1 = fig.add_subplot(gs[0])
ax1.plot(nvda.index, nvda['Close'], color=COLOR_NVDA, linewidth=1.5)
ax1.fill_between(nvda.index, nvda['Close'], alpha=0.08, color=COLOR_NVDA)
ax1.set_title('Prix de clôture ($)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Prix ($)')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))

# ── Panel 2 : Rendements journaliers ──
ax2 = fig.add_subplot(gs[1])
colors_bar = [COLOR_NVDA if r >= 0 else COLOR_VOL60 for r in nvda['Return'].fillna(0)]
ax2.bar(nvda.index, nvda['Return'] * 100, color=colors_bar, width=1, alpha=0.8)
ax2.axhline(0, color='white', linewidth=0.6, linestyle='--')
ax2.set_title('Rendements journaliers (%)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Rendement (%)')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))

# ── Panel 3 : Volatilité rolling annualisée ──
ax3 = fig.add_subplot(gs[2])
ax3.plot(nvda.index, nvda['Vol_30'] * 100, color=COLOR_VOL30, linewidth=1.2, label='Volatilité 30j', alpha=0.9)
ax3.plot(nvda.index, nvda['Vol_60'] * 100, color=COLOR_VOL60, linewidth=1.8, label='Volatilité 60j')
ax3.fill_between(nvda.index, nvda['Vol_60'] * 100, alpha=0.1, color=COLOR_VOL60)
ax3.axhline(nvda['Vol_60'].mean() * 100, color='white', linestyle='--', linewidth=0.8,
            label=f"Moyenne : {nvda['Vol_60'].mean()*100:.1f}%")
ax3.set_title('Volatilité annualisée — Rolling 30j & 60j (%)', fontsize=11, fontweight='bold')
ax3.set_ylabel('Volatilité (%)')
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax3.legend(fontsize=9)

for ax in [ax1, ax2, ax3]:
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

fig.suptitle('NVIDIA (NVDA) — Tableau de bord Risque & Performance (2022–2024)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. Distribution des rendements

L'analyse de la distribution permet de tester l'hypothèse de normalité des rendements — une hypothèse clé en finance quantitative. En pratique, les rendements financiers présentent souvent des **queues épaisses** (*fat tails*) et une **asymétrie négative**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
r = nvda['Return'].dropna()

# ── Histogramme + courbe normale ──
ax = axes[0]
ax.hist(r * 100, bins=80, color=COLOR_NVDA, alpha=0.7, density=True, label='Rendements NVDA')
mu, sigma = r.mean() * 100, r.std() * 100
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
ax.plot(x, stats.norm.pdf(x, mu, sigma), color='white', linewidth=1.8, linestyle='--', label='Distribution normale')
ax.axvline(mu, color=COLOR_VOL30, linewidth=1.2, linestyle='-', label=f'Moyenne : {mu:.2f}%')
ax.set_title('Distribution des rendements journaliers', fontsize=11, fontweight='bold')
ax.set_xlabel('Rendement (%)')
ax.set_ylabel('Densité')
ax.legend(fontsize=9)

# ── Q-Q Plot ──
ax2 = axes[1]
(osm, osr), (slope, intercept, r_val) = stats.probplot(r, dist='norm')
ax2.scatter(osm, osr, color=COLOR_NVDA, alpha=0.5, s=8, label='Rendements NVDA')
ax2.plot(osm, slope * np.array(osm) + intercept, color='white', linewidth=1.5,
         linestyle='--', label='Distribution normale')
ax2.set_title('Q-Q Plot — Test de normalité', fontsize=11, fontweight='bold')
ax2.set_xlabel('Quantiles théoriques')
ax2.set_ylabel('Quantiles observés')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

# Test de normalité de Jarque-Bera
jb_stat, jb_pval = stats.jarque_bera(r.dropna())
print(f"Test de Jarque-Bera : statistique = {jb_stat:.2f}, p-value = {jb_pval:.4f}")
if jb_pval < 0.05:
    print("→ Les rendements ne suivent PAS une distribution normale (p < 0.05)")
else:
    print("→ Pas de rejet de la normalité (p ≥ 0.05)")

## Conclusion

Cette analyse quantitative de NVIDIA sur 2022–2024 met en évidence plusieurs points clés :

**Performance** — NVIDIA affiche une performance exceptionnelle sur la période, portée par la révolution de l'IA générative à partir de 2023. Cette surperformance se reflète dans les rendements journaliers moyens positifs.

**Régimes de volatilité** — L'analyse rolling identifie clairement deux régimes distincts :
- **2022** : période de forte volatilité coïncidant avec la correction des valeurs tech et la remontée des taux
- **2023–2024** : volatilité plus maîtrisée malgré une hausse des prix, signe d'une tendance haussière soutenue

**Distribution des rendements** — Le test de Jarque-Bera confirme que les rendements de NVDA ne suivent pas une distribution normale : les queues épaisses et l'asymétrie observées sont typiques des actifs à forte croissance. Ce résultat est important pour la gestion du risque : les modèles basés sur la normalité (ex. VaR paramétrique) sous-estiment les pertes extrêmes.

**Prochaine étape →** Projet 3 : Modélisation de stratégies de tendance — Golden Cross & Death Cross